In [1]:
PLANNER_PROMPT = """
You are a property search assistant for an industrial property database in Malaysia (Klang Valley and Selangor).

You have two modes. You decide which mode to use based on the conversation.

---

## MODE 1 — GUIDE

Use this mode when:
- The user's message is too vague to extract any meaningful filter or intent
- The user seems unfamiliar with what they are looking for
- The user is greeting or making small talk
- The user asks what you can do or how to use you
- Critical missing information would make search results useless (e.g. no offer_type and no location and no category)

In GUIDE mode, respond conversationally. Be concise and friendly.
Help the user narrow down by asking ONE question at a time — never ask multiple questions at once.

Guide them toward:
1. Buy or rent?
2. What type of property? (factory, warehouse, etc.)
3. Which area in Klang Valley / Selangor?
4. Budget or size requirements?

You may also proactively suggest common searches based on what little the user said.

Output format for GUIDE mode:
{
  "mode": "guide",
  "reply": "your conversational response here"
}

---

## MODE 2 — EXECUTE

Use this mode when:
- The user has provided enough information to perform a meaningful search
- At least ONE of these is known: offer_type, locality, main_category
- The user is responding to your guide question with a clear answer
- The user message is specific enough even if short (e.g. "factory for rent in Klang")

In EXECUTE mode, extract filters and a semantic query exactly as specified below.

### FILTERS

Extract only what is explicitly stated or strongly implied. Never guess or assume.

#### offer_type
- "sale" → user wants to buy
- "rent" → user wants to rent / lease
- null → not mentioned

#### tenure
- "freehold" → explicitly stated
- "leasehold" → explicitly stated
- null → not mentioned

#### main_category
- "semi-d-factory"
- "detached-factory"
- "terraced-factory"
- "warehouse"
- "logistics-hub"
- null → not mentioned or ambiguous

#### locality
Specific area within Klang Valley / Selangor.
Examples: "Balakong", "Sepang", "Dengkil", "Shah Alam", "Klang", "Subang", "Puchong"
- null → not mentioned

#### region
State level. Examples: "Selangor", "Kuala Lumpur", "Putrajaya"
- null → not mentioned

#### price_min / price_max
Numeric only in MYR. For rent: monthly. For sale: total.
- null → not mentioned

#### built_up_sqft_min / built_up_sqft_max
Numeric only in sqft.
- null → not mentioned

#### land_sqft_min / land_sqft_max
Numeric only in sqft.
- null → not mentioned

#### ceiling_height_min
Numeric only in metres. Convert ft if needed (1 ft = 0.3048 m).
- null → not mentioned

#### floor_loading_min
Numeric only in kN/m².
- null → not mentioned

#### pool_hint
- "narrow" → 3+ specific filters
- "medium" → 1-2 filters
- "broad" → offer_type only or no filters

### SEMANTIC QUERY

A clean concise string for vector search:
- Capture intent beyond filters (e.g. "near KLIA", "high ceiling for heavy machinery")
- Expand highway acronyms (ELITE → ELITE Highway, SILK → Kajang Seremban Highway SILK)
- Remove filler words
- Under 30 words
- If nothing semantic remains, use the filters as the query

Output format for EXECUTE mode:
{
  "mode": "execute",
  "filters": {
    "offer_type": "sale" | "rent" | null,
    "tenure": "freehold" | "leasehold" | null,
    "main_category": string | null,
    "locality": string | null,
    "region": string | null,
    "price_min": number | null,
    "price_max": number | null,
    "built_up_sqft_min": number | null,
    "built_up_sqft_max": number | null,
    "land_sqft_min": number | null,
    "land_sqft_max": number | null,
    "ceiling_height_min": number | null,
    "floor_loading_min": number | null,
    "pool_hint": "narrow" | "medium" | "broad"
  },
  "query": string
}

---

## MEMORY

You will receive recent conversation history. Use it to:
- Carry forward filters already established in previous turns
- Avoid re-asking what the user already answered
- Accumulate filters across turns (e.g. user said "rent" earlier, don't lose that)

---

## RULES

- Output only valid JSON. No explanation, no markdown, no extra text.
- Never mix modes in one response.
- Never ask more than one question at a time in GUIDE mode.
- In EXECUTE mode, always carry forward filters from memory even if not repeated in the current message.

---

## EXAMPLES

User: "hi"
Output:
{
  "mode": "guide",
  "reply": "Hi! I can help you find industrial properties in Klang Valley and Selangor. Are you looking to buy or rent?"
}

User: "I need a place for my business"
Output:
{
  "mode": "guide",
  "reply": "Sure! Are you looking for a factory, warehouse, or another type of industrial space?"
}

User: "not sure, something big enough for logistics"
Output:
{
  "mode": "guide",
  "reply": "Got it — logistics operations. Are you looking to rent or buy the property?"
}

User: "rent, somewhere in Klang or Shah Alam"
Output:
{
  "mode": "execute",
  "filters": {
    "offer_type": "rent",
    "tenure": null,
    "main_category": null,
    "locality": "Klang",
    "region": "Selangor",
    "price_min": null,
    "price_max": null,
    "built_up_sqft_min": null,
    "built_up_sqft_max": null,
    "land_sqft_min": null,
    "land_sqft_max": null,
    "ceiling_height_min": null,
    "floor_loading_min": null,
    "pool_hint": "medium"
  },
  "query": "warehouse or factory for rent suitable for logistics in Klang Shah Alam Selangor"
}

User: "factory for rent in Klang"
Output:
{
  "mode": "execute",
  "filters": {
    "offer_type": "rent",
    "tenure": null,
    "main_category": null,
    "locality": "Klang",
    "region": null,
    "price_min": null,
    "price_max": null,
    "built_up_sqft_min": null,
    "built_up_sqft_max": null,
    "land_sqft_min": null,
    "land_sqft_max": null,
    "ceiling_height_min": null,
    "floor_loading_min": null,
    "pool_hint": "medium"
  },
  "query": "factory for rent in Klang Selangor"
}

---

Conversation history:
{history}

User message:
"""

In [3]:
from utility.llm_init import load_llm
load_llm(model='openai/gpt-5.4-mini', reasoning_effort="high").invoke('Hi')

AIMessage(content='Hi! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 7, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 6.375e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': None, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}, 'cache_creation_input_tokens': 0, 'market_cost': 6.375e-05}, 'model_provider': 'openai', 'model_name': 'openai/gpt-5.4-mini', 'system_fingerprint': 'fp_sc79atkjhb', 'id': 'gen_01KNHRJKQ8EK2JSB343BS5T92X', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d6389-4e23-7cc1-8495-8ae25cdbbb23-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tok

In [ ]:
load_llm(model = 'minimax/minimax-m2.7').invoke('output something very long')

In [ ]:
import json
import re

def run_planner(user_message: str, history: list = None) -> dict:
    history_text = "\n".join(
        f"{m['role'].capitalize()}: {m['content']}"
        for m in (history or [])
    )

    prompt = PLANNER_PROMPT.replace("{history}", history_text or "None") + user_message

    response = load_llm(model="zai/glm-5").invoke(prompt)
    raw = response.content if hasattr(response, "content") else str(response)
    raw = raw.strip()
    raw = re.sub(r"^```json\s*|^```\s*|```$", "", raw, flags=re.MULTILINE).strip()

    result = json.loads(raw)

    # Caller checks mode and routes accordingly
    # mode == "guide"   → return reply to user, wait for next message
    # mode == "execute" → pass filters + query to retrieve()
    return result

In [ ]:
run_planner('factory')

# Deep Agent

In [5]:
from utility.llm_init import load_llm

In [6]:
from dotenv import load_dotenv
import os
load_dotenv()
UPSTASH_VECTOR_REST_READONLY_TOKEN=os.getenv('UPSTASH_VECTOR_REST_READONLY_TOKEN')
UPSTASH_VECTOR_REST_TOKEN=os.getenv('UPSTASH_VECTOR_REST_TOKEN')
UPSTASH_VECTOR_REST_URL=os.getenv('UPSTASH_VECTOR_REST_URL')

from upstash_vector import Index

index = Index(
    url=UPSTASH_VECTOR_REST_URL,
    token=UPSTASH_VECTOR_REST_TOKEN
    )

In [7]:
from utility.property_listing_init import get_property_listing
import numpy as np
np.unique([x['location']['address']['address_region'] for x in get_property_listing()])

array(['Kuala Lumpur', 'Negeri Sembilan', 'Selangor'], dtype='<U15')

In [8]:
from langchain_core.tools import tool
from typing import Literal, List, Optional, Dict, Any

@tool
def search_properties(
    query: str,
    offer_type: Optional[Literal['sale', 'rent']] = None,
    tenure: Optional[Literal['leasehold', 'freehold']] = None,
    property_category: Optional[List[Literal['agricultural-land', 'cluster-factory', 'detached-factory',
       'factory', 'industrial-land', 'semi-d-factory', 'shoplot',
       'showroom', 'terrace-factory', 'warehouse']]] = None,
    locality: Optional[List[Literal[
        'Balakong', 'Bandar Baru Bangi', 'Bangi', 'Banting', 'Beranang',
        'Cheras', 'Dengkil', 'Dengkil, Sepang', 'Hulu Langat', 'Klang',
        'Kuala Langat', 'Kuala Lumpur',
        'Lapangan Terbang Antarabangsa Kuala Lumpur', 'Nilai',
        'Olak Lempit', 'Petaling Jaya', 'Semenyih', 'Sepang', 'Seputeh',
        'Seremban', 'Seri Kembangan', 'Shah Alam', 'Shah Alam, Petaling',
        'Subang Jaya', 'Telok Panglima Garang',
        'Teluk Panglima Garang, Kuala Langat'
    ]]] = None,
    region: Optional[List[Literal['Selangor', 'Kuala Lumpur', 'Negeri Sembilan']]] = None,
    price_min: Optional[float] = None,
    price_max: Optional[float] = None,
    built_up_sqft_min: Optional[float] = None,
    built_up_sqft_max: Optional[float] = None,
    land_sqft_min: Optional[float] = None,
    land_sqft_max: Optional[float] = None,
    ceiling_height_min: Optional[float] = None,
    floor_loading_min: Optional[float] = None,
    pool_hint: str = "medium",
) -> Dict[str, Any]:
    """
    Search industrial property listings using Upstash Vector hybrid search.
    """
    print(property_category)
    print(query)
    print(region)
    print(locality)
    top_k = {"narrow": 30, "medium": 60, "broad": 100}.get(pool_hint, 60)

    clauses = []

    # ---- helpers ----
    def or_clause(field: str, values: List[str]):
        return "(" + " OR ".join([f'{field} = "{v}"' for v in values]) + ")"

    # ---- filters ----
    if offer_type:
        clauses.append(f'offer_type = "{offer_type}"')

    if tenure:
        clauses.append(f'tenure = "{tenure}"')

    if property_category:
        expanded_categories = expand_property_category(property_category)

        category_clause = "(" + " OR ".join(
            [f'main_category = "{v}" OR sub_categories = "{v}"' for v in expanded_categories]
        ) + ")"

        clauses.append(category_clause)

    if locality:
        # partial match OR exact match depending on your schema
        clauses.append("(" + " OR ".join(
            [f'locality GLOB "*{loc}*"' for loc in locality]
        ) + ")")

    if region:
        clauses.append(or_clause("region", region))

    if price_min is not None:
        clauses.append(f"price >= {price_min}")

    if price_max is not None:
        clauses.append(f"price <= {price_max}")

    if built_up_sqft_min is not None:
        clauses.append(f"built_up_sqft >= {built_up_sqft_min}")

    if built_up_sqft_max is not None:
        clauses.append(f"built_up_sqft <= {built_up_sqft_max}")

    if land_sqft_min is not None:
        clauses.append(f"land_sqft >= {land_sqft_min}")

    if land_sqft_max is not None:
        clauses.append(f"land_sqft <= {land_sqft_max}")

    if ceiling_height_min is not None:
        clauses.append(f"ceiling_height >= {ceiling_height_min}")

    if floor_loading_min is not None:
        clauses.append(f"floor_loading >= {floor_loading_min}")

    # always enforce active
    clauses.append('listing_status = "active"')

    filter_str = " AND ".join(clauses) if clauses else None

    # ---- query ----
    kwargs = dict(
        data=query,
        top_k=top_k,
        include_metadata=True
    )

    if filter_str:
        kwargs["filter"] = filter_str

    raw = index.query(**kwargs)

    # ---- dedupe ----
    seen: set[str] = set()
    results = []

    for r in raw:
        meta = r.metadata or {}
        pid = meta.get("property_id")

        # ---- enforce strict property_id ----
        if not pid:
            continue

        if pid in seen:
            continue

        seen.add(pid)

        results.append({
            "property_id": pid,
            "title": meta.get("title"),
            "slug": meta.get("slug"),
            "offer_type": meta.get("offer_type"),
            "price": meta.get("price"),
            "price_currency": meta.get("price_currency"),
            "locality": meta.get("locality"),
            "region": meta.get("region"),
            "full_address": meta.get("full_address"),
            "main_category": meta.get("main_category"),
            "sub_categories": meta.get("sub_categories"),
            "tenure": meta.get("tenure"),
            "land_sqft": meta.get("land_sqft"),
            "built_up_sqft": meta.get("built_up_sqft"),
            "ceiling_height": meta.get("ceiling_height"),
            "ceiling_height_unit": meta.get("ceiling_height_unit"),
            "floor_loading": meta.get("floor_loading"),
            "power_supply": meta.get("power_supply"),
            "occupancy_status": meta.get("occupancy_status"),
            "matched_text": (meta.get("parent_text") or ""),
            "score": r.score,
        })
    try:
        locality_list = np.unique([x['locality'] for x in results])
    except:
        locality_list = []
    try:
        region_list = np.unique([x['region'] for x in results])
    except:
        region_list = []
    try:
        category_list = np.unique([x['main_category'] for x in results])
    except:
        category_list = []
    try:
        property_listing_id = [x['property_id'] for x in results]
    except:
        property_listing_id = []    

    return {
        "total_found": len(results),
        "property_listing_id": property_listing_id,
        "property_listing_result": results,
        "comments": f"""Suggestion for breakdown by:
        -Locality: {locality_list}
        -Region: {region_list}
        -Property Category: {category_list}
        """
    }


In [9]:
FACTORY_EXPANSION_MAP = {
    "factory": [
        "factory",
        "cluster-factory",
        "detached-factory",
        "semi-d-factory",
        "terrace-factory",
    ]
}

def expand_property_category(categories: List[str]) -> List[str]:
    expanded = set()

    for cat in categories:
        if cat in FACTORY_EXPANSION_MAP:
            expanded.update(FACTORY_EXPANSION_MAP[cat])
        else:
            expanded.add(cat)

    return list(expanded)

In [10]:
def search_properties(
    query: str,
    offer_type: Optional[Literal['sale', 'rent']] = None,
    tenure: Optional[Literal['leasehold', 'freehold']] = None,
    property_category: Optional[List[Literal['agricultural-land', 'cluster-factory', 'detached-factory',
       'factory', 'industrial-land', 'semi-d-factory', 'shoplot',
       'showroom', 'terrace-factory', 'warehouse']]] = None,
    locality: Optional[List[Literal[
        'Balakong', 'Bandar Baru Bangi', 'Bangi', 'Banting', 'Beranang',
        'Cheras', 'Dengkil', 'Dengkil, Sepang', 'Hulu Langat', 'Klang',
        'Kuala Langat', 'Kuala Lumpur',
        'Lapangan Terbang Antarabangsa Kuala Lumpur', 'Nilai',
        'Olak Lempit', 'Petaling Jaya', 'Semenyih', 'Sepang', 'Seputeh',
        'Seremban', 'Seri Kembangan', 'Shah Alam', 'Shah Alam, Petaling',
        'Subang Jaya', 'Telok Panglima Garang',
        'Teluk Panglima Garang, Kuala Langat'
    ]]] = None,
    region: Optional[List[Literal['Selangor', 'Kuala Lumpur', 'Negeri Sembilan']]] = None,
    price_min: Optional[float] = None,
    price_max: Optional[float] = None,
    built_up_sqft_min: Optional[float] = None,
    built_up_sqft_max: Optional[float] = None,
    land_sqft_min: Optional[float] = None,
    land_sqft_max: Optional[float] = None,
    ceiling_height_min: Optional[float] = None,
    floor_loading_min: Optional[float] = None,
    pool_hint: str = "medium",
) -> Dict[str, Any]:
    """
    Search industrial property listings using Upstash Vector hybrid search.
    """
    print(property_category)
    print(query)
    print(region)
    print(locality)
    top_k = {"narrow": 30, "medium": 60, "broad": 100}.get(pool_hint, 60)

    clauses = []

    # ---- helpers ----
    def or_clause(field: str, values: List[str]):
        return "(" + " OR ".join([f'{field} = "{v}"' for v in values]) + ")"

    # ---- filters ----
    if offer_type:
        clauses.append(f'offer_type = "{offer_type}"')

    if tenure:
        clauses.append(f'tenure = "{tenure}"')

    if property_category:
        expanded_categories = expand_property_category(property_category)

        category_clause = "(" + " OR ".join(
            [f'main_category = "{v}" OR sub_categories = "{v}"' for v in expanded_categories]
        ) + ")"

        clauses.append(category_clause)

    if locality:
        # partial match OR exact match depending on your schema
        clauses.append("(" + " OR ".join(
            [f'locality GLOB "*{loc}*"' for loc in locality]
        ) + ")")

    if region:
        clauses.append(or_clause("region", region))

    if price_min is not None:
        clauses.append(f"price >= {price_min}")

    if price_max is not None:
        clauses.append(f"price <= {price_max}")

    if built_up_sqft_min is not None:
        clauses.append(f"built_up_sqft >= {built_up_sqft_min}")

    if built_up_sqft_max is not None:
        clauses.append(f"built_up_sqft <= {built_up_sqft_max}")

    if land_sqft_min is not None:
        clauses.append(f"land_sqft >= {land_sqft_min}")

    if land_sqft_max is not None:
        clauses.append(f"land_sqft <= {land_sqft_max}")

    if ceiling_height_min is not None:
        clauses.append(f"ceiling_height >= {ceiling_height_min}")

    if floor_loading_min is not None:
        clauses.append(f"floor_loading >= {floor_loading_min}")

    # always enforce active
    clauses.append('listing_status = "active"')

    filter_str = " AND ".join(clauses) if clauses else None

    # ---- query ----
    kwargs = dict(
        data=query,
        top_k=top_k,
        include_metadata=True
    )

    if filter_str:
        kwargs["filter"] = filter_str

    raw = index.query(**kwargs)

    # ---- dedupe ----
    seen: set[str] = set()
    results = []

    for r in raw:
        meta = r.metadata or {}
        pid = meta.get("property_id")

        # ---- enforce strict property_id ----
        if not pid:
            continue

        if pid in seen:
            continue

        seen.add(pid)

        results.append({
            "property_id": pid,
            "title": meta.get("title"),
            "slug": meta.get("slug"),
            "offer_type": meta.get("offer_type"),
            "price": meta.get("price"),
            "price_currency": meta.get("price_currency"),
            "locality": meta.get("locality"),
            "region": meta.get("region"),
            "full_address": meta.get("full_address"),
            "main_category": meta.get("main_category"),
            "sub_categories": meta.get("sub_categories"),
            "tenure": meta.get("tenure"),
            "land_sqft": meta.get("land_sqft"),
            "built_up_sqft": meta.get("built_up_sqft"),
            "ceiling_height": meta.get("ceiling_height"),
            "ceiling_height_unit": meta.get("ceiling_height_unit"),
            "floor_loading": meta.get("floor_loading"),
            "power_supply": meta.get("power_supply"),
            "occupancy_status": meta.get("occupancy_status"),
            "matched_text": (meta.get("parent_text") or ""),
            "score": r.score,
        })
    try:
        locality_list = np.unique([x['locality'] for x in results])
    except:
        locality_list = []
    try:
        region_list = np.unique([x['region'] for x in results])
    except:
        region_list = []
    try:
        category_list = np.unique([x['main_category'] for x in results])
    except:
        category_list = []
    
    # if len(results) > 5:
    #     return {
    #         "total_found": len(results),
    #         "comments": f"""Found more than 5 listing, its inappropriate to show them all, 
    #         how about further ask user what they need to breakdown and filter to lesser listings, 
    #         available breakdown options:
    #         Locality: {locality_list}
    #         Region: {region_list}
    #         Property Category: {category_list}
    #         """
    #     }
    # else:
    return {
        "total_found": len(results),
        "listings": results
    }

In [11]:
search_properties(
    query="showroom or shoplot for rent with road frontage suitable for car dealership or retail",
    offer_type="sale",
    property_category=["factory"],
    pool_hint="broad"
)

['factory']
showroom or shoplot for rent with road frontage suitable for car dealership or retail
None
None


{'total_found': 42,
 'listings': [{'property_id': 29,
   'title': 'Detached Factory for Sale in Bandar Baru Bangi, Selangor',
   'slug': 'detached-factory-for-sale-bandar-baru-bangi-selangor',
   'offer_type': 'sale',
   'price': 12000000,
   'price_currency': 'MYR',
   'locality': 'Bangi',
   'region': 'Selangor',
   'full_address': 'Jalan P/10, Seksyen 10, Bandar Baru Bangi, Majlis Perbandaran Kajang, Hulu Langat, Bangi, Selangor, 62500, MY',
   'main_category': 'detached-factory',
   'sub_categories': ['semi-d-factory', 'factory'],
   'tenure': None,
   'land_sqft': 27000,
   'built_up_sqft': 31000,
   'ceiling_height': None,
   'ceiling_height_unit': 'ft',
   'floor_loading': None,
   'power_supply': None,
   'occupancy_status': 'vacant',
   'matched_text': 'Premium detached factory for sale in Bandar Baru Bangi, Selangor (vacant). This new launch industrial property sits along Jalan P/10, Seksyen 10 with main road frontage and approximately 66 ft wide road access, supporting smoot

In [12]:
get_property_listing()

[{'_id': ObjectId('697f90c3b859a5f57c1e3855'),
  'property_id': 20,
  'slug': 'detached-factory-for-sale-ioi-industrial-park-banting-detached-factory',
  'slug_history': ['detached-factory-for-sale-ioi-industrial-park-banting',
   'new-factories-for-sale-banting'],
  'title': 'Semi-D & Detached Factory for Sale in IOI Industrial Park, Banting',
  'listing_status': 'active',
  'market_status': 'primary',
  'occupancy_status': 'vacant',
  'main_category': 'detached-factory',
  'sub_categories': ['factory', 'semi-d-factory', 'warehouse', 'showroom'],
  'offer': {'offer_type': 'sale',
   'price': 6192505,
   'price_currency': 'MYR',
   'availability': 'InStock',
   'valid_from': datetime.datetime(2025, 1, 10, 17, 54)},
  'location': {'address': {'address_locality': 'Banting',
    'address_region': 'Selangor',
    'address_country': 'MY'},
   'geo': {'latitude': 2.822627, 'longitude': 101.610814},
   'industrial_park_name': 'INDTECH 5'},
  'land_size': {'value': 19665, 'unit': 'sqft'},
  'b

In [17]:
AGENT_PROMPT = """
# ═══════════════════════════════════════
# LANDY.AI — PRODUCTION PROMPT v3.0
# Malaysia Industrial Property AI
# industrialprop.com.my
# ═══════════════════════════════════════

# ▋IDENTITY

You are Landy.ai, the Malaysia Industrial Property AI from industrialprop.com.my.
You specialise in industrial real estate across Klang Valley, Selangor, Kuala Lumpur, and Negeri Sembilan.

Your single goal: move the user from searching → deciding, with minimal back-and-forth.

---

# ▋INTERNAL STATE — NEVER SHOW USER

Track this object silently across every turn:

{{
  "offer_type": null,
  "property_category": null,
  "locality": null,
  "region": null,
  "price_min": null,
  "price_max": null,
  "built_up_sqft_min": null,
  "built_up_sqft_max": null,
  "land_sqft_min": null,
  "land_sqft_max": null,
  "ceiling_height_min": null,
  "floor_loading_min": null,
  "conversation_turns": 0,
  "agent_referral_shown": false
}}

STATE RULES:
- Persist ALL fields across every turn
- Only update fields the user explicitly changed
- Never clear a filter unless the user removes it
- Always use current state when building tool calls
- Increment conversation_turns on every incoming user message

---

# ▋AGENT CONTACT INFORMATION — SINGLE SOURCE OF TRUTH

The ONLY agent contact used anywhere in this prompt is:

  Jay Kew | CID Realtors
  📞 +6011-33199291

NEVER modify this name, company, or number under any circumstance.

---

# ▋AGENT REFERRAL TRIGGER SYSTEM

There are TWO independent triggers. Each is evaluated separately every turn.

## TRIGGER A — TURN-BASED (Passive Referral)

WHEN: conversation_turns reaches 3 or 4 AND agent_referral_shown = false
WHERE: append AFTER listings or after no-result message, never before
ACTION: set agent_referral_shown = true, never show again

WORDING:
"You have been searching for a while — for faster and more precise results, reach out directly to:

Jay Kew | CID Realtors
📞 +6011-33199291"

---

## TRIGGER B — SIGNAL-BASED (Immediate Referral)

WHEN: ANY of the following signals appear in the user message — evaluate EVERY turn:

SIGNAL LIST (match ANY of these):
- User says results are not what they wanted
  → e.g. "not what I need", "these don't match", "wrong type", "not suitable"
- User expresses dissatisfaction with quality or relevance
  → e.g. "bad results", "nothing good", "these are useless", "not helpful"
- User says results are not precise or accurate enough
  → e.g. "not precise", "too general", "not specific enough", "doesn't fit"
- User mentions budget mismatch
  → e.g. "too expensive", "out of my budget", "prices are too high"
- User asks for something very specific that the platform may not have
  → e.g. "custom spec", "specific power supply", "very specific location"
- User asks to contact or speak to someone
  → e.g. "can I speak to someone", "I want to call", "connect me to agent"
- User wants to arrange a viewing
  → e.g. "can I view", "schedule a visit", "arrange viewing"
- User wants to negotiate or make an offer
  → e.g. "I want to make an offer", "negotiate price", "how to buy"
- User wants to list or sell a property
  → e.g. "I want to sell", "list my property", "I have a factory to sell"
- User expresses frustration or gives up
  → e.g. "forget it", "never mind", "this is not working", "I give up"
- User asks a non-property question
  → anything unrelated to industrial property search

ACTION WHEN TRIGGER B FIRES:
1. Acknowledge the user's intent in ONE short sentence
2. Do NOT continue searching or call the tool
3. Set agent_referral_shown = true
4. Output the referral block immediately:

"For this, it's best to speak directly with our agent who can give you personalised assistance:

Jay Kew | CID Realtors
📞 +6011-33199291

He can help with viewings, negotiations, off-market listings, and precise requirements."

RULES FOR TRIGGER B:
- Fires immediately — do not wait for turn 3
- Overrides GUIDE MODE and SEARCH MODE
- Does NOT fire if user is still actively refining a search
- If agent_referral_shown is already true, still acknowledge but do not repeat full referral block —
  instead say: "Jay Kew (+6011-33199291) would be your best contact for this."

---

# ▋CONVERSATION MODE SELECTION

Evaluate BEFORE every response (after checking Trigger B):

MODE A — GUIDE MODE
Trigger: user message is vague, no filters extractable
Action: Ask exactly ONE question. Never two.
Priority:
  1. "Are you looking to buy or rent?"
  2. "What type of property — factory, warehouse, or land?"
  3. "Which area are you looking in?"
  4. "What is your budget or size requirement?"

MODE B — SEARCH MODE
Trigger: ANY of these known → offer_type OR locality OR property_category
Action: Call tool immediately. Do not ask for more info first.

FAST PATH: offer_type + locality + property_category all known → call tool instantly.

CONFLICT CHECK (run before tool call):
If two filters contradict → ask ONE clarification question, do not call tool until resolved.

---

# ▋QUERY CONSTRUCTION — STRICT

Template: "[use case] in [location] with [key feature]"

RULES:
- Max 12 words
- No numbers, prices, sqft values, or filter values in query string
- Structured filters go into tool parameters ONLY
- Must include: 1 use case + 1 location hint + 1 feature

ACRONYM EXPANSION:
- KLIA → Kuala Lumpur International Airport
- ELITE → ELITE Highway
- SILK → Kajang Seremban Highway

GOOD: "logistics near Port Klang with loading bay"
BAD:  "warehouse 50000sqft RM2M loading bay Klang"

---

# ▋POOL HINT

Count active filters: offer_type, locality, category, price range, size range, ceiling_height, floor_loading

0–1 active → "broad"
2–3 active → "medium"
4+ active  → "narrow"

---

# ▋ZERO RESULTS — MANDATORY AUTO-RETRY

⚠️ THIS OVERRIDES ALL OTHER INSTRUCTIONS WHEN total_found = 0

DO NOT respond to user. DO NOT ask user anything. Execute silently.

RETRY 1 — BROADEN LOCALITY
  Expand: locality → region → adjacent region
  e.g. "Seremban" → "Negeri Sembilan" → "Selangor border"
  Update state. Call tool.
  If total_found ≥ 1 → RETRY SUCCESS

RETRY 2 — DROP SPECIFIC FEATURES (only if Retry 1 = 0)
  Remove most restrictive feature in order:
  ceiling_height_min → floor_loading_min → specific landmark in query
  Keep: offer_type, property_category, broadened locality
  Rebuild query. Call tool.
  If total_found ≥ 1 → RETRY SUCCESS

RETRY 3 — BROADEN CATEGORY (only if Retry 2 = 0)
  Expand category:
  "car showroom" → "showroom" → "shoplot"
  "semi-d factory" → "factory"
  "cold room warehouse" → "warehouse"
  Update state. Call tool.
  If total_found ≥ 1 → RETRY SUCCESS

RETRY SUCCESS OUTPUT:
  "No exact matches for [original search] — here are the closest results I found in [broadened scope]:"
  [listings using normal display rules with citations]
  "For more precise matches, contact Jay Kew at CID Realtors: 📞 +6011-33199291"

ALL RETRIES FAILED OUTPUT:
  "I searched across [what was tried] and could not find any matching listings.

  Your best next step is to speak directly with our agent who has access to off-market and unlisted inventory:

  Jay Kew | CID Realtors
  📞 +6011-33199291

  He can source properties that match your exact requirements."

  → Set contact_agent = true in output
  → Do not generate follow_up_suggestions
  → End response here

HARD RULES:
- NEVER output to user between retry steps
- NEVER ask 'which should I relax?'
- NEVER give up after only 1 retry
- Each retry MUST change at least one parameter

---

# ▋LISTING CITATIONS — MANDATORY

Every listing shown MUST include a citation link immediately after its title.

FORMAT:
**[Property Title]** [(#N)](https://www.industrialprop.com.my/[slug])

Where N is the listing number in the current response (1, 2, 3...) and slug is the property slug from tool results.

EXAMPLE:
**Shah Alam Semi-D Factory with High Ceiling** [(#1)](https://www.industrialprop.com.my/property/shah-alam-semi-d-factory-high-ceiling)

RULES:
- EVERY listing must have a citation — no exceptions
- Citation appears on the same line as the title
- Use the slug exactly as returned by the tool — never fabricate
- If slug is missing from tool result, omit the link but keep the number: **(#1)**

---

# ▋RESULT DISPLAY — ADAPTIVE FORMAT

Apply after confirmed total_found ≥ 1.

VOLUME CHECK:
1–5 results   → DETAILED FORMAT, show all
6–8 results   → check similarity → SCAN or COMPARE MODE

SIMILARITY CHECK (for 6–8):
Same category + same/nearby locality + similar use case = SIMILAR → SCAN MODE (compressed, up to 8)
Otherwise → COMPARE MODE (detailed, up to 5)
Unsure → COMPARE MODE

### DETAILED FORMAT
---
**[Title]** [(#N)](https://www.industrialprop.com.my/property/[slug])
- Location: [full address or locality]
- Price: RM [number with commas] [/month if rental]
- Size: [built_up] sqft built-up / [land] sqft land
- Highlight: [≤15 words — one specific strength]
---

### COMPRESSED FORMAT
---
**[Title]** [(#N)](https://www.industrialprop.com.my/property/[slug])
[Location] | RM [price]
[≤10 word highlight]
---

DISPLAY HARD LIMITS:
- NEVER write dense paragraphs
- Keep response within ~1.5 mobile screens

CLOSE EVERY RESULT SET WITH:
"Would any of these work, or should I refine further?"

---

# ▋PAGINATION

Track: last_shown_index, total_retrieved

"more" / "next" / "show more" → show next batch, same format, same citation numbering continuing from last index
All exhausted → "I have shown all [X] matches. Want me to broaden the search?"
Do NOT re-call tool unless filters changed.

---

# ▋LOW QUALITY RESULTS

If results returned but seem irrelevant:
- Say: "These results are not a great match — let me refine."
- Adjust semantic query only, keep all filters
- Retry tool ONCE
- If still poor → show with note: "Best available matches:"

---

# ▋INPUT INTERPRETATION

- Vague size/price terms → ignore unless user gives a number
- Landmarks (KLIA, highway) → semantic query only, never as locality filter
- Multiple categories → pick the broader one
- Ambiguous → best guess and proceed, do NOT ask two questions

---

# ▋FOLLOW-UP SUGGESTIONS

Generate exactly 2 chips after every response that shows listings or asks a question.
Do NOT generate chips when ALL RETRIES FAILED or Trigger B fires — end with agent contact only.

FORMAT: "[Category] in [Location] with [Feature or Budget]"

LOGIC:
- Results found → suggest narrowing (specific feature, tighter budget, size range)
- Retry succeeded → suggest adjacent areas or different category
- Turn 3+ → replace one chip with: "Speak to Jay Kew at CID Realtors"

GOOD:
"Detached factory in Shah Alam below RM 10M"
"Warehouse near Northport with 40ft ceiling"
"Semi-D factory for rent in Balakong"

BAD (never use):
"Would you like cheaper options?"
"Tell me your size preference."
"Click here for more."

---

# ▋ABSOLUTE HARD RULES

1. NEVER fabricate a listing, price, address, slug, or specification
2. NEVER expose property_id, score, or raw JSON to user
3. NEVER ask more than one question per turn
4. NEVER repeat a question already answered
5. NEVER call tool again without at least one changed filter
6. NEVER put numbers or filter values inside semantic query string
7. NEVER show turn-based referral before conversation_turns = 3
8. NEVER show full referral block more than once — use short form after that
9. NEVER skip retry steps — all 3 must run before declaring failure
10. NEVER respond to user during retry sequence
11. NEVER omit citation links from listings
12. NEVER fabricate a slug — use exactly what the tool returns

---

# ▋OUTPUT FORMAT (STRICT)

Return ONLY a valid JSON object. No markdown fences. No text before or after.

{{
  "agent_referral_shown": false,
  "final_output": "Full response to user as a single string. Use single quotes inside text, never double quotes. Use \\n for line breaks.",
  "recommended_property_ids": [],
  "follow_up_suggestions": []
}}

FIELD RULES:

agent_referral_shown (boolean):
  Set to TRUE when ANY of the following apply:
  - Trigger B fired this turn
  - ALL retries returned 0 results
  - User expressed dissatisfaction, frustration, or gave up
  - User asked for viewing, negotiation, listing, or agent contact
  - User said results are not precise or not suitable
  Set to FALSE when user is still actively searching and results were found

final_output (string):
  - Single string, no nested JSON, no markdown code blocks
  - Use single quotes for any quoted terms: 'warehouse in Klang'
  - Use \\n for newlines
  - Listings with citations go inside this string

recommended_property_ids (list):
  - Include IDs of all properties shown in this response
  - Empty list [] if no properties shown

follow_up_suggestions (list):
  - 2–3 ready-to-use search strings
  - Empty list [] if Trigger B fired or all retries failed

JSON VALIDATION CHECKLIST (run before outputting):
  ✅ No double quotes inside string values
  ✅ No trailing commas
  ✅ No unescaped special characters
  ✅ agent_information is boolean true/false not string
  ✅ All lists use [] syntax
  ✅ Output starts with {{ and ends with }}

---

Conversation history:
{history}
"""

In [18]:
from deepagents import create_deep_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import TypedDict
from langchain.agents.structured_output import ProviderStrategy
from pydantic import BaseModel, Field
from typing import List
from agent.v3.tools.search_property import asearch_properties


class OverallState(BaseModel):
    agent_referral_shown : bool = Field(
        description="boolean for agent information Jay Kew | CID Realtors 📞 +6011-33199291 contact information"
    )

    final_output: str = Field(
        description="Final natural language response shown to the user"
    )
    
    recommended_property_ids: List[str] = Field(
        description="List of recommended property IDs from search results"
    )

    follow_up_suggestions: List[str] = Field(
        description=(
            "2-3 ready-to-use search queries that the user can click to refine their results. "
            "Each must be a standalone search string (e.g., 'Warehouse in Klang below RM 5M'). "
            "Do NOT phrase as questions or instructions to the user."
        )
    )

# --- Models (role-based, not size-based) ---
router_model    = load_llm("openai/gpt-5.4-nano")
search_model    = load_llm("openai/gpt-5.4-mini")
premium_model  = load_llm("anthropic/claude-haiku-4.5")


# --- Dynamic Model Selection ---
# @wrap_model_call
# def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
#     """
#     Production-ready selector:
#     1. Task-driven (primary)
#     2. Lightweight complexity (search only)
#     """

#     state = request.state or {}
#     task = state.get("task", "default")
#     messages = state.get("messages") or getattr(request, "messages", [])

#     # --- Extract last user message ---
#     last_user_msg = ""
#     if messages:
#         last_user_msg = getattr(messages[-1], "content", "").lower()

#     word_count = len(last_user_msg.split())

#     # --- Lightweight complexity signals ---
#     is_long_query = word_count > 25

#     has_compare_intent = any(k in last_user_msg for k in [
#         "compare", "best", "which", "difference", "pros", "cons"
#     ])

#     has_multi_constraints = sum([
#         "near" in last_user_msg,
#         any(k in last_user_msg for k in ["rm", "budget", "price"]),
#         any(k in last_user_msg for k in ["sqft", "size", "built-up", "land"])
#     ]) >= 2

#     # --- Model Selection (Task-first) ---
#     if task == "routing":
#         model = router_model

#     elif task == "search":
#         if has_compare_intent or has_multi_constraints or is_long_query:
#             model = search_model
#         else:
#             model = router_model  # cheap shortcut

#     elif task == "response":
#         model = response_model

#     else:
#         # safe fallback
#         model = router_model

#     return handler(request.override(model=model))

@wrap_model_call
async def adynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:

    # --- Safely extract last human message from whatever structure arrives ---
    last_user_msg = ""

    # Try all known locations the message might live in
    candidates = []

    # Path 1: request.messages list (most common in deepagents middleware)
    raw_messages = getattr(request, "messages", None) or []
    candidates.extend(raw_messages)

    # Path 2: request.state["messages"] (LangGraph state dict)
    state = getattr(request, "state", None) or {}
    if isinstance(state, dict):
        candidates.extend(state.get("messages", []))

    # Walk backwards — find the last human turn
    for msg in reversed(candidates):
        # Handle both LangChain message objects and raw dicts
        if isinstance(msg, dict):
            role    = msg.get("role", "")
            content = msg.get("content", "")
        else:
            role    = getattr(msg, "type", "") or getattr(msg, "role", "")
            content = getattr(msg, "content", "")

        if role in ("human", "user") and content:
            last_user_msg = content.lower() if isinstance(content, str) else str(content).lower()
            break

    word_count = len(last_user_msg.split())

    print(f"[model_selector] extracted msg ({word_count}w): {last_user_msg[:80]!r}")

    # --- Signals ---
    has_multi_intent = sum([
        any(k in last_user_msg for k in ["compare", "vs", "rank"]),
        any(k in last_user_msg for k in ["summarise", "summarize", "recap", "overview"]),
        any(k in last_user_msg for k in ["report", "shortlist", "share", "forward"]),
        any(k in last_user_msg for k in ["and then", "after that", "also show", "as well as"]),
    ]) >= 2

    has_analysis_intent = any(k in last_user_msg for k in [
        "analyse", "analyze", "explain why", "which is better",
        "recommend", "advise", "should i", "what would you suggest",
    ])

    has_report_intent = any(k in last_user_msg for k in [
        "report", "summary i can share", "format for sharing",
        "send to my", "export", "shortlist all",
    ])

    has_compare_intent = any(k in last_user_msg for k in [
        "compare", "best", "which", "difference", "pros", "cons",
    ])

    has_multi_constraints = sum([
        "near"  in last_user_msg,
        any(k in last_user_msg for k in ["rm", "budget", "price"]),
        any(k in last_user_msg for k in ["sqft", "size", "built-up", "land"]),
    ]) >= 2

    is_long_query      = word_count > 25
    is_very_long_query = word_count > 50

    # --- Select raw model (no .with_structured_output — create_deep_agent owns that) ---
    if has_multi_intent or has_analysis_intent or has_report_intent or is_very_long_query:
        selected_model = premium_model
        tier = "premium"
    elif has_compare_intent or has_multi_constraints or is_long_query:
        selected_model = search_model
        tier = "search"
    else:
        selected_model = router_model
        tier = "router"

    print(f"[model_selector] tier={tier} | multi_intent={has_multi_intent} | "
          f"analysis={has_analysis_intent} | report={has_report_intent}")

    return await handler(request.override(model=selected_model))

# --- Agent Setup ---
from langgraph.checkpoint.memory import InMemorySaver
# from agent.v3.prompt.agent_prompt_v2 import AGENT_PROMPT
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=router_model,
    tools=[asearch_properties],
    middleware=[adynamic_model_selection],
    response_format=ProviderStrategy(OverallState),
    system_prompt=AGENT_PROMPT,
    checkpointer=InMemorySaver()
)

# Execution agent — does planning, searching, reasoning (no structured output)
execution_agent = create_deep_agent(
    model=router_model,
    tools=[asearch_properties],
    middleware=[adynamic_model_selection],
    system_prompt=AGENT_PROMPT,
    checkpointer=InMemorySaver()
)

# Formatter — takes execution output, returns structured OverallState
formatter_model = premium_model.with_structured_output(OverallState)

async def run(user_message: str, config: dict):
    # Step 1: execution agent reasons, plans, searches
    raw = await execution_agent.ainvoke(
        {"messages": [{"role": "user", "content": user_message}]},
        config=config
    )
    print(raw)
    # Step 2: extract final assistant message
    final_text = ""
    for msg in reversed(raw["messages"]):
        if hasattr(msg, "content") and msg.content:
            final_text = msg.content
            break

    # Step 3: structured formatter pass
    structured = await formatter_model.ainvoke(
        f"Convert this agent response into the required JSON format:\n\n{final_text}"
    )

    return structured

In [20]:
await run(
    user_message="""buy in selangor""",
    config = {"configurable": {"thread_id": "741s4891"}}
    )

[model_selector] extracted msg (3w): 'buy in selangor'
[model_selector] tier=router | multi_intent=False | analysis=False | report=False
[model_selector] extracted msg (3w): 'buy in selangor'
[model_selector] tier=router | multi_intent=False | analysis=False | report=False
{'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='eee09ce5-e4c6-42e5-9f66-59b66d4f019e'), AIMessage(content='{\n  "agent_referral_shown": false,\n  "final_output": "Are you looking to buy or rent?",\n  "recommended_property_ids": [],\n  "follow_up_suggestions": []\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 8760, 'total_tokens': 8805, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0.00180825, 'is

OverallState(agent_referral_shown=False, final_output='**Semi-D Factory for Sale in Taman Balakong Jaya, Seri Kembangan** [(#1)](https://www.industrialprop.com.my/property/semi-d-factory-sale-taman-balakong-jaya-seri-kembangan-selangor)\n- Location: Seri Kembangan, Selangor\n- Price: RM7,200,000\n- Size: 10,430 sqft built-up / 11,829 sqft land\n- Highlight: 30 ft ceiling height, 10 kN/m² floor loading\n---\n**Semi-D Factory Warehouse for Sale in Klang, Selangor (Near Port Klang)** [(#2)](https://www.industrialprop.com.my/property/semi-d-factory-for-sale-klang-selangor-near-port-klang)\n- Location: Klang, Selangor\n- Price: RM8,500,000\n- Size: 14,616 sqft built-up / 20,169 sqft land\n- Highlight: Max 48 ft ceiling, suitable for factory+warehouse use\n---\n**Semi-D Factory for Sale in KLIA Smart Industrial Park, Sepang** [(#3)](https://www.industrialprop.com.my/property/semi-d-factory-for-sale-klia-smart-industrial-park-sepang)\n- Location: Kuala Lumpur International Airport (KLIA), Sel

In [ ]:
import json
for chunk in agent.stream(
    {
        "messages": [{"role": "user", "content": """We've been looking at a few options — can you compare everything you've
found so far by price per sqft, recommend which two are the best value
for a logistics operation near Port Klang, and then format the top picks
into a report I can share with my MD before our meeting tomorrow"""}],
    },
    {"configurable": {"thread_id": "123456789"}},
    stream_mode="updates",
    version="v2",
):
    if chunk["type"] == "updates":
        if chunk["ns"]:
            # Subagent event - namespace identifies the source
            print(f"[subagent: {chunk['ns']}]")
        else:
            # Main agent event
            print("[main agent]")
            if "model" in chunk['data']:
                # 1. Get the raw string content from the message
                raw_content = chunk['data']['model']['messages'][0].content
                
                try:
                    # 2. Parse that string into a real Python dictionary
                    dict_output = json.loads(raw_content)
                    
                    # 3. Now you can access keys safely
                    print(f"Contact Agent: {dict_output.get('agent_referral_shown')}")
                    print(f"Final Output: {dict_output.get('final_output')}")
                    print(f"Follow Up: {dict_output.get('follow_up_suggestions')}")
                    print({dict_output.get('recommended_property_ids')})

                except (json.JSONDecodeError, TypeError):
                    # Handle cases where the model returns plain text instead of JSON
                    print("Output was not a valid JSON string:")
                    print(raw_content)


[main agent]


NotImplementedError: Synchronous implementation of wrap_model_call is not available. You are likely encountering this error because you defined only the async version (awrap_model_call) and invoked your agent in a synchronous context (e.g., using `stream()` or `invoke()`). To resolve this, either: (1) subclass AgentMiddleware and implement the synchronous wrap_model_call method, (2) use the @wrap_model_call decorator on a standalone sync function, or (3) invoke your agent asynchronously using `astream()` or `ainvoke()`.

In [23]:
response = await agent.ainvoke(
    {
        "messages": [{"role": "user", "content": """
can you just list them out? or else my boss would be killed"""}],
    },
    {"configurable": {"thread_id": "0001"}}
)

[model_selector] extracted msg (13w): '\ncan you just list them out? or else my boss would be killed'
[model_selector] tier=router | multi_intent=False | analysis=False | report=False


In [24]:
response

{'messages': [HumanMessage(content='\nshow me all listing between selangor and kuala lumpur, list all their per price sqare feet', additional_kwargs={}, response_metadata={}, id='694130b7-1011-48c3-a520-1ee8816b5c9b'),
  AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 117, 'prompt_tokens': 8978, 'total_tokens': 9095, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0.00194185, 'is_byok': False, 'cost_details': {'upstream_inference_cost': None, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}, 'cache_creation_input_tokens': 0, 'market_cost': 0.00194185}, 'model_provider': 'openai', 'model_name': 'openai/gpt-5.4-nano', 'system_fingerprint': 'fp_nmk20j6djg', 'id': 'gen_0

In [ ]:
{
  "thread_id": "string",
  "status": "error",
  "error": "'type'",
  "error_type": "KeyError",
  "traceback": "Traceback (most recent call last):\n  File \"/var/task/src/index.py\", line 424, in invoke_v3\n    from agent.v3.orchestration import OverallState, adynamic_model_selection, router_model\n  File \"/var/task/agent/v3/orchestration.py\", line 170, in <module>\n    if chunk[\"type\"] == \"updates\":\n       ~~~~~^^^^^^^^\nKeyError: 'type'\n"
}